In [2]:
%pip install datatrove[all] xxhash

  Using cached vllm-0.27.1-cp38-abi3-manylinux_2_28_x86_64.whl.metadata (11 kB)
Using cached vllm-0.27.1-cp38-abi3-manylinux_2_28_x86_64.whl (312.9 MB)
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
os.environ["HF_TOKEN"] = ""

In [11]:
import time
import tempfile
from datatrove.pipeline.readers import HuggingFaceDatasetReader
from datatrove.pipeline.writers import JsonlWriter, HuggingFaceDatasetWriter
from datatrove.pipeline.dedup import MinhashDedupSignature, MinhashDedupBuckets, MinhashDedupCluster, MinhashDedupFilter
from datatrove.pipeline.dedup.minhash import MinhashConfig
from datatrove.pipeline.filters.base_filter import BaseFilter
from datatrove.executor import LocalPipelineExecutor

dataset_read = "salyamq/kz_ds_v1"
dataset_write = "salyamq/kk-corpus-v1"

CPU_CORES = 49
NUM_SHARDS = 270  
BAD_CHARS = {chr(0x058A), chr(0x058E), chr(0x05AA), chr(0x05AB), chr(0x05B0), chr(0x05B1), chr(0x05B2), chr(0x05B4), 
             chr(0x05B5), chr(0x05B6), chr(0x05B7), chr(0x05B8), chr(0x05B9), chr(0x05BC), chr(0x05BE), chr(0x05BF), 
             chr(0x05C0), chr(0x05C1), chr(0x05C3), chr(0x05F3), chr(0x05F4), chr(0x0600), chr(0x0605), chr(0x060B), 
             chr(0x060C), chr(0x060F), chr(0x061B), chr(0x061F), chr(0x064B), chr(0x064C), chr(0x064D), chr(0x064E), 
             chr(0x064F), chr(0x0650), chr(0x0651), chr(0x0652), chr(0x0653), chr(0x0656), chr(0x0657), chr(0x065E), 
             chr(0x066A), chr(0x066B), chr(0x066D), chr(0x0670), chr(0x06D6), chr(0x06D7), chr(0x06D9), chr(0x06DA), 
             chr(0x06DD), chr(0x06DE), chr(0x06E1), chr(0x06E2), chr(0x06E3), chr(0x06E9), chr(0x06EA), chr(0x06EB), 
             chr(0x06EC), chr(0x06ED), chr(0x07A6), chr(0x07A7), chr(0x07A8), chr(0x07AA), chr(0x07AB), chr(0x07AC), 
             chr(0x07AD), chr(0x07B0), chr(0x07F8), chr(0x0820), chr(0x0826), chr(0x082C), chr(0x082E), chr(0x0836), 
             chr(0x085C), chr(0x085E), chr(0x086B), chr(0x086D), chr(0x0901), chr(0x0902), chr(0x0903), chr(0x093C), 
             chr(0x093E), chr(0x093F), chr(0x0940), chr(0x0941), chr(0x0942), chr(0x0943), chr(0x0947), chr(0x0948), 
             chr(0x0949), chr(0x094A), chr(0x094B), chr(0x094C), chr(0x094D), chr(0x0964), chr(0x0981), chr(0x0982), 
             chr(0x09BC), chr(0x09BE), chr(0x09BF), chr(0x09C0), chr(0x09C1), chr(0x09C7), chr(0x09C8), chr(0x09CB), 
             chr(0x09CD), chr(0x0A0C), chr(0x0A3E), chr(0x0A3F), chr(0x0A40), chr(0x0A70), chr(0x0A82), chr(0x0ABA), 
             chr(0x0ABC), chr(0x0ABE), chr(0x0AC0), chr(0x0AC1), chr(0x0AC4), chr(0x0AC6), chr(0x0ACA), chr(0x0ACB), 
             chr(0x0ACC), chr(0x0ADA), chr(0x0ADC), chr(0x0AE2), chr(0x0B3C), chr(0x0B3F), chr(0x0BBE), chr(0x0BBF), 
             chr(0x0BC0), chr(0x0BC1), chr(0x0BC2), chr(0x0BC7), chr(0x0BC8), chr(0x0BCA), chr(0x0BCD), chr(0x0C11), 
             chr(0x0C3E), chr(0x0C3F), chr(0x0C41), chr(0x0C42), chr(0x0C46), chr(0x0C4D), chr(0x0C82), chr(0x0CBE), 
             chr(0x0CBF), chr(0x0CC0), chr(0x0CC1), chr(0x0CC2), chr(0x0CC6), chr(0x0CC7), chr(0x0CC8), chr(0x0CCB), 
             chr(0x0CCD), chr(0x0D02), chr(0x0D3E), chr(0x0D3F), chr(0x0D42), chr(0x0D4D), chr(0x0D82), chr(0x0DCA), 
             chr(0x0DCF), chr(0x0DD2), chr(0x0E31), chr(0x0E34), chr(0x0E35), chr(0x0E36), chr(0x0E37), chr(0x0E38), 
             chr(0x0E39), chr(0x0E3F), chr(0x0E47), chr(0x0E48), chr(0x0E49), chr(0x0E4C), chr(0x0E4D), chr(0x0E4F), 
             chr(0x0EB9), chr(0x0F0B), chr(0x0F0D), chr(0x0F3A), chr(0x0F3B), chr(0x0F71), chr(0x0F72), chr(0x0F74), 
             chr(0x0F7A), chr(0x0F7C), chr(0x0F97), chr(0x0FB1), chr(0x0FB2), chr(0x0FB3), chr(0x102B), chr(0x102C), 
             chr(0x102D), chr(0x102E), chr(0x102F), chr(0x1030), chr(0x1031), chr(0x1032), chr(0x1036), chr(0x1037), 
             chr(0x1038), chr(0x1039), chr(0x103A), chr(0x103B), chr(0x103C), chr(0x103D), chr(0x103E), chr(0x1064), 
             chr(0x1083), chr(0x1086), chr(0x1087), chr(0x108F), chr(0x17B6), chr(0x17B7), chr(0x17B8), chr(0x17BB), 
             chr(0x17BC), chr(0x17C1), chr(0x17C2), chr(0x17C3), chr(0x17CA), chr(0x17D2), chr(0x19F5), chr(0x1A17), 
             chr(0x1A18), chr(0x1B42), chr(0x1BA5), chr(0x1BAA), chr(0x1C4A), chr(0x1DC8), chr(0x3001), chr(0x3002), 
             chr(0x3008), chr(0x3009), chr(0x300A), chr(0x300B), chr(0x300C), chr(0x300D), chr(0x300E), chr(0x300F), 
             chr(0x3010), chr(0x3011), chr(0x3012), chr(0x3015), chr(0x3016), chr(0x3017), chr(0x301C), chr(0x3020), 
             chr(0x3030), chr(0x303D), chr(0x309A), chr(0x30FB), chr(0xA673), chr(0xA789), chr(0xA78A), chr(0xA9C1), 
             chr(0xA9C2), chr(0xABE3), chr(0xABE4), chr(0xABE9), chr(0xE003), chr(0xE02D), chr(0xE04E), chr(0xE056), 
             chr(0x3486E), chr(0xC8783), chr(0xC878A), chr(0xC8830), chr(0xC89BA), chr(0xC8A0A), chr(0xC8A16), chr(0xC8A19), 
             chr(0xC8B00), chr(0xC8B04), chr(0xC8B0A), chr(0xC8B23), chr(0xC8B24), chr(0xC8B52), chr(0x100003)}
print(f"chars loaded: {len(BAD_CHARS)}")

chars loaded: 279


In [15]:
class BadCharsFilter(BaseFilter):
    name = "bad chars filter"

    def __init__(self, bad_chars: set[str], exclusion_writer=None):
        super().__init__(exclusion_writer=exclusion_writer)
        self.bad_chars = bad_chars

    def filter(self, doc) -> bool:
        # true document alive, false left
        return not (self.bad_chars & set(doc.text))

In [16]:
reader = HuggingFaceDatasetReader(
    dataset=dataset_read,
    dataset_options={
        "split": "train",
        "data_dir": "data",
    },
    text_key="text",
    id_key="id",
)

badchars_removed_writer = JsonlWriter(output_folder="removed_badchars")
minhash_removed_writer = JsonlWriter(output_folder="removed_minhash")
final_writer = HuggingFaceDatasetWriter(dataset=dataset_write, private=True, local_working_dir=tempfile.gettempdir())

In [17]:
config = MinhashConfig(n_grams=5, num_buckets=16, hashes_per_bucket=8)
print(f"permutations: {config.num_buckets * config.hashes_per_bucket}")

permutations: 128


In [21]:
start = time.perf_counter()

LocalPipelineExecutor(
    pipeline=[
        reader,
        BadCharsFilter(bad_chars=BAD_CHARS, exclusion_writer=badchars_removed_writer),
        JsonlWriter(output_folder="clean_text", expand_metadata = True),  
    ],
    tasks=NUM_SHARDS,
    workers=CPU_CORES,
).run()

print(f"stage 0 (bad chars) done: {time.perf_counter() - start} sec")

2026-08-23 09:45:03.023 | INFO     | datatrove.utils.logging:add_task_logger:76 - Launching pipeline for rank=7
2026-08-23 09:45:03.025 | INFO     | datatrove.utils.logging:log_pipeline:108 - 
--- 🛠️ PIPELINE 🛠
📖 - READER: 🤗 HuggingFace
🔻 - FILTER: bad chars filter
💽 - WRITER: 🐿 Jsonl
Generating train split: 100%|██████████| 13340376/13340376 [08:46<00:00, 25337.66 examples/s]
2026-08-23 09:55:43.824 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 1/270 tasks completed.
2026-08-23 09:55:43.834 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 2/270 tasks completed.
2026-08-23 09:55:44.527 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 3/270 tasks completed.
2026-08-23 09:55:44.562 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 4/270 tasks completed.
2026-08-23 09:55:45.374 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 5/270 tasks completed.
2026-08-23 09:55:45.568 | INFO     | datatrove.executor.local:_la

stage 0 (bad chars) done: 1240.6699810880236 sec


In [22]:
from datatrove.pipeline.readers import JsonlReader

clean_reader = JsonlReader(data_folder="clean_text")
start = time.perf_counter()

LocalPipelineExecutor(
    pipeline=[clean_reader, MinhashDedupSignature(output_folder="sigs", config=config)],
    tasks=NUM_SHARDS,
    workers=CPU_CORES,
).run()

print(f"stage 1signatures: {time.perf_counter() - start}sec")

2026-08-23 10:06:34.672 | INFO     | datatrove.utils.logging:add_task_logger:76 - Launching pipeline for rank=12
2026-08-23 10:06:34.672 | INFO     | datatrove.utils.logging:log_pipeline:108 - 
--- 🛠️ PIPELINE 🛠
📖 - READER: 🐿 Jsonl
🫂 - DEDUP: 🎯 MinHash stage 1
2026-08-23 10:06:34.762 | INFO     | datatrove.pipeline.readers.base:read_files_shard:206 - Reading input file 00012.jsonl.gz, 1/1
2026-08-23 10:26:46.171 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 1/270 tasks completed.
2026-08-23 10:26:48.947 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 2/270 tasks completed.
2026-08-23 10:26:54.563 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 3/270 tasks completed.
2026-08-23 10:26:55.825 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 4/270 tasks completed.
2026-08-23 10:26:58.523 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 5/270 tasks completed.
2026-08-23 10:26:58.992 | INFO     | datatrove.execut

stage 1signatures: 7852.688386847032sec


In [24]:
start = time.perf_counter()

LocalPipelineExecutor(
    pipeline=[MinhashDedupBuckets(input_folder="sigs", output_folder="buckets", config=config)],
    tasks=config.num_buckets,
    workers=CPU_CORES,
).run()

print(f"stage 2 buckets done: {time.perf_counter() - start}s")

2026-08-23 12:25:15.568 | INFO     | datatrove.utils.logging:add_task_logger:76 - Launching pipeline for rank=7
2026-08-23 12:25:15.568 | INFO     | datatrove.utils.logging:log_pipeline:108 - 
--- 🛠️ PIPELINE 🛠
🫂 - DEDUP: 🎯 MinHash stage 2
2026-08-23 12:25:15.574 | INFO     | datatrove.pipeline.dedup.minhash:run:401 - Running worker 1/1 on bucket 007. Hash range: [0, np.uint64(2305843009213693951)]
2026-08-23 12:25:15.688 | INFO     | datatrove.pipeline.dedup.minhash:run:456 - Finished initializing signatures priority queue.
2026-08-23 12:26:50.668 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 1/16 tasks completed.
2026-08-23 12:26:51.370 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 2/16 tasks completed.
2026-08-23 12:26:51.568 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 3/16 tasks completed.
2026-08-23 12:26:51.574 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 4/16 tasks completed.
2026-08-23 12:26:51.903 | INF

stage 2 buckets done: 99.39064774988219s


In [26]:
print(config.num_buckets)        
print(config.hashes_per_bucket)  
print(config.hash_config)        

16
8
HashConfig(precision=64, hash_fc=xxhash)


In [27]:
start = time.perf_counter()

LocalPipelineExecutor(
    pipeline=[MinhashDedupCluster(input_folder="buckets", output_folder="clusters", config=config)],
    tasks=1,
).run()

print(f"stage 3 cluster done: {time.perf_counter() - start}s")

2026-08-23 12:31:19.172 | INFO     | datatrove.utils.logging:add_task_logger:76 - Launching pipeline for rank=0
2026-08-23 12:31:19.225 | INFO     | datatrove.utils.logging:log_pipeline:108 - 
--- 🛠️ PIPELINE 🛠
🫂 - DEDUP: 🎯 MinHash stage 3
2026-08-23 12:31:19.227 | INFO     | datatrove.pipeline.dedup.minhash:run:561 - Loading dup files...
Reading dup files: 100%|██████████| 16/16 [02:39<00:00,  9.95s/it]
2026-08-23 12:33:58.460 | INFO     | datatrove.pipeline.dedup.minhash:run:571 - Finished reading dup files.
2026-08-23 12:35:12.571 | SUCCESS  | datatrove.executor.base:_run_for_rank:140 - Processing done for rank=0
2026-08-23 12:35:12.580 | INFO     | datatrove.executor.base:_run_for_rank:147 - 

📉📉📉 Stats: Task 0 📉📉📉

Total Runtime: 3 minutes and 53.30 seconds

🫂 - DEDUP: 🎯 MinHash stage 3
    Runtime: (100.00%) 3 minutes and 53.30 seconds [3 minutes, 53 seconds and 296.07 milliseconds±0 milliseconds/doc]
    Stats: {duplicates: 6970572, cluster_size: 6970572 [min=2, max=21773, 3.00±

stage 3 cluster done: 233.58192035299726s


In [40]:
from huggingface_hub import HfApi

api = HfApi()


api.delete_repo(
    repo_id="salyamq/kk-corpus-v1",
    repo_type="dataset"
)

In [41]:
from huggingface_hub import create_repo

create_repo(
    repo_id="salyamq/kk-corpus-v1",
    private=True,
    repo_type="dataset",
    exist_ok=True
)

RepoUrl('https://huggingface.co/datasets/salyamq/kk-corpus-v1', endpoint='https://huggingface.co', repo_type='dataset', repo_id='salyamq/kk-corpus-v1')

In [44]:
import shutil
from pathlib import Path
from datatrove.pipeline.writers.parquet import ParquetWriter
from huggingface_hub import HfApi


out_root = Path("stage4_parquet")
clean_dir = out_root / "clean"
removed_dir = out_root / "removed"

clean_dir.mkdir(parents=True, exist_ok=True)
removed_dir.mkdir(parents=True, exist_ok=True)

final_writer = ParquetWriter(
    output_folder=str(clean_dir),
    output_filename="${rank}.parquet",
)

minhash_removed_writer = ParquetWriter(
    output_folder=str(removed_dir),
    output_filename="${rank}.parquet",
)

start = time.perf_counter()

LocalPipelineExecutor(
    pipeline=[
        clean_reader,
        MinhashDedupFilter(
            input_folder="clusters",
            exclusion_writer=minhash_removed_writer,
        ),
        final_writer,
    ],
    tasks=NUM_SHARDS,
    workers=36,
).run()

print(f"stage 4 (filter + push) done: {time.perf_counter() - start}sec")

2026-08-23 13:47:05.575 | INFO     | datatrove.utils.logging:add_task_logger:76 - Launching pipeline for rank=14
2026-08-23 13:47:05.576 | INFO     | datatrove.utils.logging:log_pipeline:108 - 
--- 🛠️ PIPELINE 🛠
📖 - READER: 🐿 Jsonl
🫂 - DEDUP: 🎯 MinHash stage 4
💽 - WRITER: 📒 Parquet
2026-08-23 13:47:05.635 | INFO     | datatrove.pipeline.readers.base:read_files_shard:206 - Reading input file 00014.jsonl.gz, 1/1
2026-08-23 13:47:23.042 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 1/270 tasks completed.
2026-08-23 13:47:23.248 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 2/270 tasks completed.
2026-08-23 13:47:23.544 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 3/270 tasks completed.
2026-08-23 13:47:23.659 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 4/270 tasks completed.
2026-08-23 13:47:23.703 | INFO     | datatrove.executor.local:_launch_run_for_rank:81 - 5/270 tasks completed.
2026-08-23 13:47:23.792 | INFO 

stage 4 (filter + push) done: 186.32260619592853sec


In [45]:
api = HfApi()

api.upload_folder(
    folder_path=str(clean_dir),
    repo_id=dataset_write,
    repo_type="dataset",
    path_in_repo="data",
    commit_message="clean parquet shards",
    allow_patterns="*.parquet",
)

CommitInfo(commit_url='https://huggingface.co/datasets/salyamq/kk-corpus-v1/commit/1f8f9741ed8f90b354a56c289f587316e0c7b47c', commit_message='clean parquet shards (part 2)', commit_description='', oid='1f8f9741ed8f90b354a56c289f587316e0c7b47c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/salyamq/kk-corpus-v1', endpoint='https://huggingface.co', repo_type='dataset', repo_id='salyamq/kk-corpus-v1'), pr_revision=None, pr_num=None)

In [47]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [49]:
import pyarrow.parquet as pq
from pathlib import Path

def count_parquet(folder):
    total = 0
    for path in Path(folder).glob("*.parquet"):
        table = pq.read_table(path)
        total += len(table)
    return total


clean_count = count_parquet("stage4_parquet/clean")
minhash_removed = count_parquet("stage4_parquet/removed")

print(f"clean docs: {clean_count}")
print(f"deleted dedup (minhash): {minhash_removed}")
print(f"total processed: {clean_count + minhash_removed}")

clean docs: 8664472
deleted dedup (minhash): 4649617
total processed: 13314089


In [50]:
import gzip
from pathlib import Path

def count_jsonl_gz(folder):
    total = 0
    for path in Path(folder).glob("*.jsonl.gz"):
        with gzip.open(path, "rt", encoding="utf-8") as f:
            total += sum(1 for _ in f)
    return total

badchars_removed = count_jsonl_gz("removed_badchars")
print(f"deleted bad chars: {badchars_removed}")

deleted bad chars: 26287
